# INFOMED Playwright Scraper

Notebook para recolher os dados da tabela do INFOMED com Playwright, guardar em CSV e JSON, e servir como base para carga posterior numa base de dados SQLite/Drift.

## Objetivo
- Aceder à tabela do INFOMED.
- Extrair campos estruturados por linha.
- Exportar os resultados para ficheiros locais.

## Saídas
- `medicamentos_infomed.csv`
- `medicamentos_infomed.json`

In [1]:
# IMPORTS

import csv
import json
from pathlib import Path
from typing import Iterable

from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeoutError

In [ ]:
# TABLE EXTRACTION FROM INFOMED PORTAL

OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = OUTPUT_DIR / "medicamentos_infomed.csv"
JSON_PATH = OUTPUT_DIR / "medicamentos_infomed.json"

BASE_URL = "https://www.infarmed.pt/web/infarmed/servicos-on-line/pesquisa-do-medicamento"
TABLE_SELECTOR = "#form\\:tbl"
ROW_SELECTOR = "#form\\:tbl_data tr"


def clean_text(value: str | None) -> str:
    if not value:
        return ""
    return " ".join(value.replace("\xa0", " ").split()).strip()


def parse_currency(value: str | None) -> float | None:
    text = clean_text(value).replace("€", "").replace(".", "").replace(",", ".")
    if not text or text.lower() in {"preço livre", "n/a"}:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def extract_row(row) -> dict:
    def cell(selector: str) -> str:
        element = row.query_selector(selector)
        return clean_text(element.inner_text() if element else "")

    nome_element = row.query_selector(".nome-column span[id$=':nomeMed']")
    info_link = row.query_selector(".info-column a[href]")

    nome_medicamento = clean_text(nome_element.inner_text()) if nome_element else cell(".nome-column")

    return {
        "nRegisto": int(cell(".registo-column").replace("Nº registo", "") or 0),
        "dci": cell(".subs-column").replace("Substância Ativa/DCI", ""),
        "brandName": nome_medicamento.replace("Nome do medicamento", ""),
        "form": cell(".forma-column").replace("Forma farmacêutica", ""),
        "dosage": cell(".dosagem-column").replace("Dosagem", ""),
        "boxsize": cell(".tamanho-column").replace("Tamanho da embalagem", ""),
        "cnpem": int(cell(".cnpem-column").replace("CNPEM", "") or 0),
        "pricePVP": parse_currency(cell(".pvp-column").replace("Preço (PVP)", "")),
        "priceUtente": parse_currency(cell(".utente-column").replace("Preço Utente", "")),
        "pricePensionista": parse_currency(cell(".pension-column").replace("Preço Pensionistas", "")),
        "commercialized": cell(".comerc-column").replace("Comerc.", ""),
        "isGeneric": cell(".ui-helper-hidden").replace("Genérico", ""),
        "infoUrl": info_link.get_attribute("href") if info_link else "",
    }


def scrape_infomed_table() -> list[dict]:
    records: list[dict] = []

    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        page = browser.new_page(viewport={"width": 1600, "height": 1200})
        page.goto(BASE_URL, wait_until="networkidle")

        try:
            page.wait_for_selector(TABLE_SELECTOR, timeout=15000)
            page.wait_for_selector(ROW_SELECTOR, timeout=15000)
        except PlaywrightTimeoutError as exc:
            browser.close()
            raise RuntimeError(f"Não foi possível encontrar a tabela do INFOMED: {exc}") from exc

        rows = page.query_selector_all(ROW_SELECTOR)
        for row in rows:
            try:
                records.append(extract_row(row))
            except Exception as exc:
                print(f"Linha ignorada por erro de parsing: {exc}")

        browser.close()

    return records


def export_records(records: Iterable[dict]) -> None:
    records = list(records)
    if not records:
        raise RuntimeError("Nenhum medicamento foi extraído.")

    fieldnames = list(records[0].keys())
    with CSV_PATH.open("w", newline="", encoding="utf-8-sig") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)

    with JSON_PATH.open("w", encoding="utf-8") as json_file:
        json.dump(records, json_file, ensure_ascii=False, indent=2)

    print(f"Exportação concluída: {len(records)} registos")
    print(f"CSV: {CSV_PATH}")
    print(f"JSON: {JSON_PATH}")

In [ ]:
if __name__ == "__main__":
    records = scrape_infomed_table()
    export_records(records)